# GEAP Platform SDK demo — build → deploy → route

A **SDK-first** tour of the Gemini Enterprise Agent Platform on *this repo's* modules: MCP tool servers, ADK agents, deploy to Agent Runtime, A2A register/discover, and the 5-tier complexity router with real per-model cost.

Live/costly cells are opt-in via `GEAP_RUN_*` env flags (default off), so the notebook runs top-to-bottom safely. Rubric/query cells need a reachable `AGENT_ENGINE_ID` (see `.env`).

## Setup

In [ ]:
# Repo-root bootstrap so `src.*` imports resolve from notebooks/demo/
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

def enabled(flag: str) -> bool:
    # Costly/live cells are opt-in via GEAP_RUN_* env flags (default off).
    return os.environ.get(flag, '0') in ('1', 'true', 'True')

import vertexai
from src import config
vertexai.init(project=config.GCP_PROJECT_ID, location=config.GCP_REGION)
AGENT_RESOURCE = (
    f'projects/{config.GCP_PROJECT_ID}/locations/{config.GCP_REGION}'
    f'/reasoningEngines/{config.AGENT_ENGINE_ID}'
)
print('project:', config.GCP_PROJECT_ID, '| region:', config.GCP_REGION)
print('coordinator engine:', config.AGENT_ENGINE_ID)

## Phase 1 — MCP tool servers

Three custom FastMCP servers (search / booking / expense) back the agents. Deploy them to Cloud Run with `bash scripts/deploy_all.sh` or `python -m src.deploy.deploy_mcp_servers`; here we import the search tools directly to show the surface.

In [ ]:
from src.mcp_servers.search import server as search_server
# FastMCP tools are plain functions under the hood:
print(search_server.search_flights('SFO', 'JFK')[:2])
print(search_server.search_hotels('New York', max_price=350)[:2])

## Phase 2 — Build ADK agents

The **coordinator** is a domain router with `travel_agent` + `expense_agent` sub-agents (AgentTool delegation) and a `PreloadMemoryTool` for Memory Bank. The separate **5-tier router** (`src/router/agents.py`) is an economic optimizer — see Phase 6.

In [ ]:
from src.agents.coordinator_agent import coordinator_agent, root_agent
print('name:', coordinator_agent.name)
print('sub_agents:', [a.name for a in coordinator_agent.sub_agents])
print('tools:', [type(t).__name__ for t in coordinator_agent.tools])
print('before_agent_callback:', getattr(coordinator_agent, 'before_agent_callback', None).__name__
      if coordinator_agent.before_agent_callback else None)

## Phase 3 — Run coordination (live, guarded)

Query the deployed coordinator engine. Set `GEAP_RUN_QUERY=1` to actually stream (needs a reachable engine).

In [ ]:
if enabled('GEAP_RUN_QUERY'):
    from vertexai import agent_engines
    engine = agent_engines.get(AGENT_RESOURCE)
    for event in engine.stream_query(
        message='Find me a flight from SFO to JFK next Monday and check the expense policy for it.',
        user_id='demo-user',
    ):
        print(event)
else:
    print('skipped — set GEAP_RUN_QUERY=1 to stream against', AGENT_RESOURCE)

## Phase 4 — Deploy to Agent Runtime (guarded)

`run_deploy` deploys/updates coordinator, router, or all. Memory-enabled agents are wrapped in an `AdkApp` with Vertex Memory Bank + Session services automatically (see `deploy_agents._build_app`). Set `GEAP_RUN_DEPLOY=1` (~3-5 min/agent, creates billable engines).

In [ ]:
from src.deploy.deploy_agents import run_deploy
if enabled('GEAP_RUN_DEPLOY'):
    deployed = run_deploy('coordinator')          # or 'router' / 'all'
    # deployed = run_deploy('coordinator', update=True)  # update existing from .env
    print(deployed)
else:
    print('skipped — set GEAP_RUN_DEPLOY=1 to deploy (billable). '
          'CLI: uv run python -m src.deploy.deploy_agents all')

## Phase 5 — Register / discover on A2A (preview-optional)

Publish the coordinator's agent card to the Agent Registry and discover A2A agents. This is **preview-optional** — every path degrades gracefully (logs `A2A preview not enabled — skipping` and returns empty) rather than crashing. CLI: `python -m src.deploy.register_a2a` / `--discover`.

In [ ]:
from src.a2a.agent_card import build_agent_card, agent_card_dict
card = build_agent_card()
print('card name:', card.name, '| skills:', len(card.skills))
from src.registry import get_a2a_agents
print('registered A2A agents:', get_a2a_agents())  # [] if preview not enabled

> **Optional manual aside — publish to Gemini Enterprise.** Some demos publish the reasoning engine to a Gemini Enterprise app via the raw `provisionedReasoningEngine` REST surface. That flow is intentionally *not* adopted here; `register_a2a` (above) is the supported path in this repo.

## Phase 6 — Complexity routing + real multi-model cost

The router scores each prompt 0-1 and picks one of five tiers (lite → flash → sonnet → pro → opus). Cheaper tiers handle most traffic; only the hardest prompts reach Opus. `classify_complexity` is **async**.

In [ ]:
from src.router.complexity import classify_complexity, score_to_model_tier, tier_to_model
from src.eval.cost_model import per_request_cost_usd

prompts = [
    'What is the per diem for meals in NYC?',
    'Plan a 3-city EU trip under $4k, optimize layovers, and pre-check every expense against policy.',
]
for p in prompts:
    if enabled('GEAP_RUN_QUERY'):
        r = await classify_complexity(p)            # noqa: F704 (top-level await in Jupyter)
        tier = score_to_model_tier(r.score)
        print(f'{r.score:.2f} {tier:>6} ({tier_to_model(tier)}) <- {p[:48]}')
    else:
        print('skipped classify (needs model access); set GEAP_RUN_QUERY=1 —', p[:48])

In [ ]:
# Illustrative per-request cost by tier (1k in / 500 out) — priced by cost_model.
for tier in ['lite', 'flash', 'sonnet', 'pro', 'opus']:
    model = tier_to_model(tier)
    usd = per_request_cost_usd(model, 1000, 500)
    print(f'{tier:>6} {model:<28} ${usd:.5f}/req')

## Recap

L1-native SDKs (`vertexai`, `agent_engines`, ADK) carry build/deploy/query; this repo adds the 5-tier router + measured multi-model cost, memory-backed deploys, and preview-optional A2A on top. Next: `evaluation_sdk_demo.ipynb` for the eval + monitoring flywheel.